# ComplaintLens: Routing and Theme Discovery in Consumer Finance Complaints

**Text Mining & Search, University of Milano-Bicocca, A.Y. 2026/27**
Team: **Abdul Sami** and **Fatima** · Instructors: Prof. Marco Viviani, Stefano Molari

Banks and other financial companies receive thousands of written complaints every month, and someone has to read them, send them to the right team and understand what keeps going wrong. We use real complaints published by the US **Consumer Financial Protection Bureau (CFPB)** to study two questions:

1. **Task 1, complaint routing (text classification):** can we send a complaint to the right product team automatically?
2. **Task 2, complaint themes (topic modeling):** what are people actually complaining about, without reading every complaint?

For both tasks we compare a **sparse baseline** with a **dense, embedding-based alternative**, with separate train / validation / test data.

| Part | Who |
|---|---|
| Data collection, pre-processing, exploration, **Task 1** (sections 1–4) | Abdul Sami |
| **Task 2** (section 5) | Fatima |
| Results and discussion (sections 6–7) | both |

## 0. Setup

Install the requirements once from a terminal (we used Python 3.11 on Windows / Anaconda):

```bash
pip install -r requirements.txt
```

Then put the data files in `data/raw/` (see section 1) and run the notebook top to bottom.

In [1]:
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")   # silences a joblib warning on Windows

import json, random, re, time, warnings
from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, adjusted_rand_score, classification_report,
                             confusion_matrix, f1_score, normalized_mutual_info_score)
from sklearn.metrics.pairwise import linear_kernel
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.dpi"] = 110

### Configuration

We kept every setting in one dictionary, so we never had to hunt for numbers inside the functions.

In [3]:
SMOKE_TEST = False   # True = quick run on a small fake dataset, only to check the installation

CONFIG = {
    "seed": 42,
    "paths": {
        "raw_dir": Path("data/raw"),             # the CSV files of the CFPB narratives archive
        "processed_dir": Path("data/processed"),
        "results_dir": Path("results"),
        "cache_dir": Path("data/processed/cache"),
    },
    "data": {
        "date_from": "2023-01-01", "date_to": "2024-12-31",
        "n_products": 5,       # the 5 most frequent products
        "per_class": 4000,     # 5 x 4,000 = 20,000 complaints before cleaning
        "min_words": 20,       # very short narratives say almost nothing
        "split": {"train": 0.70, "val": 0.15, "test": 0.15},
        "chunksize": 200_000,  # rows per chunk, so the big files fit in memory
        "near_dup_threshold": 0.90,
        # the CFPB renamed some products over the years, so we merge the old and new names
        "product_merge": {
            "Credit reporting, credit repair services, or other personal consumer reports": "Credit reporting",
            "Credit reporting or other personal consumer reports": "Credit reporting",
            "Credit card or prepaid card": "Credit card",
        },
    },
    "preprocess": {"remove_redactions": True, "lemmatize": True, "keep_negations": True},
}

In [4]:
CONFIG["classification"] = {
    "tfidf": {"ngram_range": (1, 2), "min_df": 3, "max_df": 0.9, "sublinear_tf": True},
    "C_grid": [0.1, 1.0, 10.0],
    "use_embeddings": True,
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "batch_size": 64,
    "zeroshot": {"enabled": False, "model": "facebook/bart-large-mnli", "n_samples": 500},
}

CONFIG["topics"] = {"k_grid": [5, 8, 10, 15, 20], "n_top_words": 10, "lda_max_iter": 20,
                    "use_embeddings": True}

In [5]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
    except ImportError:
        pass


def save_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=float)


def savefig(fig, name):
    path = RES / "figures" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path)
    plt.show()


set_seed(CONFIG["seed"])
for p in CONFIG["paths"].values():
    p.mkdir(parents=True, exist_ok=True)
RES = CONFIG["paths"]["results_dir"]


### Installation checking

This writes a small fake CSV with the same columns as the CFPB files and shrinks the settings, so the whole notebook runs in a few seconds. The fake data is very easy, so the scores are meaningless; we only used it to check that everything runs.

In [11]:
if SMOKE_TEST:
    VOCAB = {
        "Credit reporting": "credit report bureau inaccurate account dispute equifax score information",
        "Debt collection": "debt collector collection call owe payment harass letter agency",
        "Credit card": "card charge interest fee statement limit purchase merchant",
        "Checking or savings account": "account deposit overdraft branch check savings fund withdraw",
        "Mortgage": "mortgage loan escrow servicer modification home foreclosure payment",
    }
    rnd = random.Random(0)
    filler = "i am writing because the company did not help me and i want this resolved quickly".split()
    rows = []
    for prod, words in VOCAB.items():
        for i in range(150):
            text = " ".join(rnd.choice(words.split() + filler) for _ in range(40)) + " on XX/XX/2023 XXXX {$100.00}"
            rows.append({"Complaint ID": f"{prod[:3]}{i}", "Date received": "2024-03-01", "Product": prod,
                         "Issue": "synthetic", "Consumer complaint narrative": text})
    fake = Path("smoke_test_tmp")
    CONFIG["paths"] = {"raw_dir": fake / "raw", "processed_dir": fake / "processed",
                       "results_dir": fake / "results", "cache_dir": fake / "cache"}
    for p in CONFIG["paths"].values():
        p.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(fake / "raw/complaints.csv", index=False)
    RES = CONFIG["paths"]["results_dir"]
    CONFIG["data"].update({"per_class": 120, "chunksize": 200})
    CONFIG["topics"].update({"k_grid": [3, 5], "lda_max_iter": 5})
    print("SMOKE TEST: fake data in", fake.resolve())

## 1. Data: building a documented subset

*Section 1–4 by Abdul Sami.*

**Source.** CFPB Consumer Complaint Database, <https://www.consumerfinance.gov/data-research/consumer-complaints/>. The narratives are written by consumers, published only with their consent, and personal details are masked as `XXXX`.

**A problem we ran into.** Our first plan was the current export (`complaints.csv.zip`, about 5 GB unzipped). After downloading it we found it has **15 columns and no `Consumer complaint narrative`**: the CFPB stopped publishing complaint narratives in **August 2026**, and the website export and the API no longer contain the text. The CFPB did publish an official **narratives archive** (complaints received from December 2011 to 14 August 2026):
<https://www.consumerfinance.gov/foia-requests/foia-electronic-reading-room/cfpb-consumer-complaint-database-narratives-archive/>

For 2023–2024 we downloaded these five ZIP files, extracted them and put the CSV files in `data/raw/`:

- `CCDB_Export_4_November_2022_through_August_2023.zip`
- `CCDB_Export_5_September_2023_through_March_2024.zip`
- `CCDB_Export_6_April_2024_through_July_2024.zip`
- `CCDB_Export_7_August_2024_through_October_2024.zip`
- `CCDB_Export_8_November_2024_through_December_2024.zip`

(My suggestion don't open these files in Excel: Excel stops at 1,048,576 rows and silently cuts the rest.)

**Why a subset.** The archive has more than a million usable complaints for 2023–2024 and is very imbalanced (credit reporting alone is about 75% of them). We keep complaints that have a narrative, were received in 2023–2024, have at least 20 words and belong to the 5 most frequent products, then draw **4,000 per product** and split **70 / 15 / 15**, stratified by product. The files are read in chunks, in two passes: the first counts complaints per product, the second samples them.

In [17]:
# the columns we need, and the short names we use in the code
COLS = {"Complaint ID": "complaint_id", "Date received": "date", "Product": "product_raw",
        "Issue": "issue", "Consumer complaint narrative": "text"}

# the archive files may use the old export names or the API names
ALIASES = {
    "Complaint ID": ["Complaint ID", "complaint_id"],
    "Date received": ["Date received", "date_received"],
    "Product": ["Product", "product"],
    "Issue": ["Issue", "issue"],
    "Consumer complaint narrative": ["Consumer complaint narrative", "complaint_what_happened",
                                     "Complaint narrative"],
}

In [16]:
def find_columns(path):
    # mapping the columns of one CSV to the names in COLS; stop if the text column is missing
    found = pd.read_csv(path, nrows=0).columns.tolist()
    mapping, missing = {}, []
    for std, options in ALIASES.items():
        hit = next((c for c in options if c in found), None)
        if hit is None:
            missing.append(std)
        else:
            mapping[hit] = std
    if missing:
        raise ValueError(f"{path.name} is missing {missing}. Columns found: {found}")
    return mapping

In [18]:
def read_chunks(cfg):
    # yields every CSV in data/raw/ chunk by chunk
    files = sorted(cfg["paths"]["raw_dir"].glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"No CSV files in {cfg['paths']['raw_dir']} (see the download list above)")
    for f in files:
        mapping = find_columns(f)
        print(f"\nreading {f.name}")
        for chunk in pd.read_csv(f, usecols=list(mapping), dtype=str, chunksize=cfg["data"]["chunksize"]):
            yield chunk.rename(columns=mapping)


def filter_chunk(chunk, cfg):
    d = cfg["data"]
    df = chunk.rename(columns=COLS)
    df = df[df["text"].notna()].copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df[(df["date"] >= d["date_from"]) & (df["date"] <= d["date_to"])]
    df["product"] = df["product_raw"].map(lambda p: d["product_merge"].get(p, p))
    return df[df["text"].str.split().str.len() >= d["min_words"]]